In [1]:
import pandas as pd
import ast
# 如果是制表符分隔
drop_df = pd.read_csv("/root/Drop.txt", sep="\t", header=None,
                  names=["id", "name", "abandon1", "abandon2", "relation"])[['id','name','relation']]
                                                                            
drop_df['relation'] = drop_df['relation'].apply(ast.literal_eval)
item_name_list = ["id", "name", "abandon1", "abandon2", "abandon3", "abandon4", "abandon5", "shape"]
for i in range(30):
    item_name_list.append("abandon"+str(i+6))
df_items =  pd.read_csv("/root/Item.txt", sep="\t", header=None,
                  names=item_name_list)[['id','name','shape','abandon7']].rename(columns={"id": "item_id","abandon7": "price"})
map_info = pd.read_csv("/root/bidking_map_priors.csv")         

def relation_to_dict(rel_list):
    # 先累加每个 key 的第五个元素
    if len(rel_list[0]) == 0:
        return {}
    sums = {}
    for sub in rel_list:
        key = sub[1]
        val = sub[4]
        sums[key] = sums.get(key, 0) + val

    total = sum(sums.values())
    # 再转成比值
    return {k: v / total for k, v in sums.items()}


# 假设 df 是你的 DataFrame，relation 列存的是 list of list
drop_df["relation_dict"] = drop_df["relation"].apply(relation_to_dict)
general_dict = {}
print(drop_df["relation_dict"].iloc[0])
for index,row in drop_df.iterrows():
    general_dict[row['id']] = row['relation_dict']
node_list = list(general_dict.keys())


def expand_node(node_id, general_dict, node_list, base_prob=1.0):
    """
    node_id: 当前节点编号
    general_dict: 外层 dict，key 是节点，value 是子节点的 dict
    node_list: 非叶子节点的集合或列表
    base_prob: 当前路径累计的概率
    返回: dict {leaf_id: prob}
    """
    result = {}

    # 如果这个节点没有子字典，说明它是叶子
    if node_id not in general_dict:
        result[node_id] = base_prob
        return result

    # 遍历子节点
    for child_id, prob in general_dict[node_id].items():
        new_prob = base_prob * prob
        if child_id in node_list:
            # 递归展开
            child_result = expand_node(child_id, general_dict, node_list, new_prob)
            # 合并结果
            for k, v in child_result.items():
                result[k] = result.get(k, 0) + v
        else:
            # 已经是叶子
            result[child_id] = result.get(child_id, 0) + new_prob

    return result

def df_wrapper(node_id, general_dict, node_list):
    result_dict = expand_node(node_id, general_dict, node_list)
    return pd.DataFrame(list(result_dict.items()), columns=["item_id", "prob"])

def get_merged_table(mapindex,general_dict,node_list,df_items):
    df_probs = df_wrapper(mapindex,general_dict, node_list)
    merged = pd.merge(df_probs, df_items, on="item_id", how="left")
    merged["rarity"] = merged["item_id"].astype(str).str[-4]
    return merged

{8001: 0.015625, 8002: 0.015625, 8003: 0.015625, 8004: 0.015625, 8005: 0.015625, 8006: 0.015625, 8007: 0.015625, 8008: 0.015625, 8009: 0.015625, 8010: 0.015625, 8011: 0.015625, 8012: 0.015625, 8013: 0.015625, 8014: 0.015625, 8015: 0.015625, 8016: 0.015625, 8017: 0.015625, 8018: 0.015625, 8019: 0.015625, 8020: 0.015625, 8021: 0.015625, 8022: 0.015625, 8023: 0.015625, 8024: 0.015625, 8025: 0.015625, 8026: 0.015625, 8027: 0.015625, 8028: 0.015625, 8029: 0.015625, 8030: 0.015625, 8031: 0.015625, 8032: 0.015625, 8033: 0.015625, 8034: 0.015625, 8035: 0.015625, 8036: 0.015625, 8037: 0.015625, 8038: 0.015625, 8039: 0.015625, 8040: 0.015625, 8041: 0.015625, 8042: 0.015625, 8043: 0.015625, 8044: 0.015625, 8046: 0.015625, 8047: 0.015625, 8051: 0.015625, 8052: 0.015625, 8053: 0.015625, 8054: 0.015625, 8055: 0.015625, 8056: 0.015625, 8057: 0.015625, 8058: 0.015625, 8059: 0.015625, 8060: 0.015625, 8061: 0.015625, 8062: 0.015625, 8063: 0.015625, 8064: 0.015625, 8065: 0.015625, 8066: 0.015625, 8067: 0

In [2]:
get_merged_table(2012,general_dict,node_list,df_items)

,item_id,prob,name,shape,price,rarity
0,1101006,0.003241,生日贺卡,11,114,1
1,1031010,0.008855,卡贴,11,115,1
2,1071002,0.002709,手机支架,11,120,1
3,1031001,0.008499,塑料发箍,11,123,1
4,1051010,0.003329,乳白弹珠,11,125,1
...,...,...,...,...,...,...
534,1076002,0.000039,全画幅相机,22,186900,6
535,1096004,0.000132,佛跳墙,11,68100,6
536,1096001,0.000092,罗曼尼康帝,12,92880,6
537,1096005,0.000067,野生大黄鱼,21,127260,6
